Before starting the project let's connect to the github repository

Now that the account is connected, let's start by importing the right libraries and gathering the data.
The first data is about traffic accident, available as a json at this address

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
url = 'https://esploradati.istat.it/SDMXWS/rest/data/41_983'
header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}
params = {
    'startPeriod' : 2020
}

In [ ]:
data = requests.get(url, headers=header, params=params, stream=True)
data.raise_for_status()

with open("database_car_accident.csv", 'w') as file:
    file.write(data.text)


Now that the first part of the data has been downloaded, we can proceed with the second part.
After downloading the file about italian cities [`https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31], let's analyze the data.

In [2]:
car_accident = pd.read_csv('data/database_car_accident.csv')
print(car_accident.info())

<class 'pandas.DataFrame'>
RangeIndex: 113856 entries, 0 to 113855
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          113856 non-null  str    
 1   FREQ              113856 non-null  str    
 2   REF_AREA          113856 non-null  int64  
 3   DATA_TYPE         113856 non-null  str    
 4   RESULT            113856 non-null  str    
 5   TIME_PERIOD       113856 non-null  int64  
 6   OBS_VALUE         113856 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

With that script we can observe that all data contained in the last 9 columns return as null, meaning they are empty. Now let's see what data the other columns have and if they are all unique

In [3]:
for name in list(car_accident)[:7]:
    print(f'For the column {name} we have these unique values: ')
    print(car_accident[name].unique())

For the column DATAFLOW we have these unique values: 
<StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str
For the column FREQ we have these unique values: 
<StringArray>
['A']
Length: 1, dtype: str
For the column REF_AREA we have these unique values: 
[  1001   1002   1003 ... 111105 111106 111107]
For the column DATA_TYPE we have these unique values: 
<StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
For the column RESULT we have these unique values: 
<StringArray>
['F', 'M', '9']
Length: 3, dtype: str
For the column TIME_PERIOD we have these unique values: 
[2020 2021 2022 2023 2024]
For the column OBS_VALUE we have these unique values: 
[    4     2     5     6     0     1     3    14     9    10     8     7
    25    46    31    36    24    18    30    23    27    20    39    32
    50    54    47    40    34    11    38    43    22    29    21    33
    12    17    13    15    16    19    58    97    80    86   102    37
    70    55    62    71    45    28    35    4

for the colums that have data inside let's download the dataflow from this endpoint: https://esploradati.istat.it/SDMXWS/rest/dataflow/IT1/41_983. 
With this data we can have an isnight on what each column and value actually means

-DATAFLOW -> Refers to the dataset id, should be safe to ignore that data 

-FREQ -> Refers to the frequency of the data (annually) 

-REF_AREA -> connected to the Codice Comune (alfanumerico) found in the second database

-DATA_TYPE -> KILLINJ stands for deceased and injured, ROADACC only for injured

-RESULT -> M is for deceased, F is for Injured and 9 is for total

In [ ]:
#RIGA 200
<structure:Code id="M">
  <common:Name xml:lang="en">killed</common:Name>
  <common:Name xml:lang="it">morto</common:Name>
</structure:Code>
<structure:Code id="F">
  <common:Name xml:lang="en">injured</common:Name>
  <common:Name xml:lang="it">ferito</common:Name>
</structure:Code>
<structure:Code id="9">
  <common:Name xml:lang="en">total</common:Name>
  <common:Name xml:lang="it">totale</common:Name>
</structure:Code>

#Riga 187.766
<structure:Code id="KILLINJ">
  <common:Name xml:lang="it">morti e feriti</common:Name>
</structure:Code>
<structure:Code id="ROADACC">
  <common:Name xml:lang="it">incidenti stradali con lesioni alle persone</common:Name>
</structure:Code>

Now let's do the same for the other database

In [102]:
#in this case I had to change the default separator to get an accurate reading
cities_2020 = pd.read_csv('data/comuni_italiani_2020.csv', sep=';')
print(cities_2020.info())


<class 'pandas.DataFrame'>
RangeIndex: 7903 entries, 0 to 7902
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   Codice Ripartizione geografica  7903 non-null   int64
 1   Codice Regione                  7903 non-null   int64
 2   Codice Provincia (Storico)      7903 non-null   int64
 3   Codice Provincia/Uts            7903 non-null   int64
 4   Codice Comune (alfanumerico)    7903 non-null   int64
 5   Codice Comune (numerico)        7903 non-null   int64
 6   Comune                          7902 non-null   str  
 7   Comune (dizione straniera)      124 non-null    str  
 8   Sigla automobilistica           7811 non-null   str  
 9   Capoluogo di Provincia/Uts      7903 non-null   int64
 10  Capoluogo di Regione            7903 non-null   int64
 11  Popolazione legale              7903 non-null   int64
 12  Anno Censimento                 7903 non-null   int64
 13  Superficie (Km

This database is different, it contains null values in 3 columns (Comune, Comune (dizione straniera) and Sigla automobilistica). 
we can safely ignore the Comune (dizione straniera) since for this analysis our interest is on the local names, while for the others two it's worth investigating a little bit

In [103]:
comune_missing_value = cities_2020[cities_2020['Comune'].isnull()]
comune_missing_value

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
164,1,1,1,201,1168,1168,NaN,NaN,TO,0,0,7998,2011,"24,6422",2020,7849,2020


In [ ]:
sigla_missing_value = cities_2020[cities_2020['Sigla automobilistica'].isnull()]
sigla_missing_value

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
5066,4,15,63,263,63001,63001,Acerra,NaN,NaN,0,0,56465,2011,"54,7117",2020,58334,2020
5067,4,15,63,263,63002,63002,Afragola,NaN,NaN,0,0,63820,2011,"17,9106",2020,61861,2020
5068,4,15,63,263,63003,63003,Agerola,NaN,NaN,0,0,7373,2011,"19,8316",2020,7640,2020
5069,4,15,63,263,63004,63004,Anacapri,NaN,NaN,0,0,6546,2011,"6,4666",2020,6940,2020
5070,4,15,63,263,63005,63005,Arzano,NaN,NaN,0,0,34933,2011,"4,7310",2020,32750,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5153,4,15,63,263,63088,63088,Visciano,NaN,NaN,0,0,4550,2011,"10,9036",2020,4226,2020
5154,4,15,63,263,63089,63089,Volla,NaN,NaN,0,0,22989,2011,"6,2063",2020,24905,2020
5155,4,15,63,263,63090,63090,Santa Maria la Carità,NaN,NaN,0,0,11726,2011,"3,9788",2020,11685,2020
5156,4,15,63,263,63091,63091,Trecase,NaN,NaN,0,0,9118,2011,"6,2131",2020,8643,2020


The missing value in the comune column is from the UTS 201, which corresponds to the city of Torino.
For the missing values in the Sigla automobilistica column, it seems they all are from the city of Napoli, so the NaN value is replaved with NA

In [6]:
comune_missing_value['Comune'] = 'Torino'
sigla_missing_value['Sigla automobilistica'] = "NA"

In [ ]:
for name in list(cities_2020):
    print(f'For the column {name} we have these unique values: ')
    print(cities_2020[name].unique())

For the column Codice Ripartizione geografica we have these unique values: 
[1 2 3 4 5]
For the column Codice Regione we have these unique values: 
[ 1  2  7  3  4  5  6  8 11  9 10 12 15 13 14 16 17 18 19 20]
For the column Codice Provincia (Storico) we have these unique values: 
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 108 109 110 111]
For the column Codice Provincia/Uts we have these unique values: 
[201   2   3   4   5   6   7   8   9 210  11  12  13  14 215  16  17  18
  19  20  21  22  23  24  25  26 227  28  29  30  31  32  33  34  35  36
 237  38  39  40  41  42  43  44  45  46  47 248  49  50  51  52  53  5

The Codice Ripartizione geografica is a range of number (from 1 to 5) used to divide italy in:
1 - North-west (Piemonte, Valle d'Aosta, Liguria, Lombardia)
2 - North-est (Trentino-Alto Adige, Veneto, Friuli-Venezia Giulia, Emilia-Romagna)
3 - Center (Toscana, Umbria, Marche, Lazio)
4 - South (Abruzzo, Molise, Campania, Puglia, Basilicata, Calabria)
5 - Islands (Sicilia, Sardegna)

The Codice Regione is an identification number (that goes from 1 to 21) set by the Agenzia delle Entrate
01 - Abruzzo
02 - Basilicata
03 - Bolzano (P.A.)
04 - Calabria
05 - Campania
06 - Emilia-Romagna
07 - Friuli-Venezia Giulia
08 - Lazio
09 - Liguria
10 - Lombardia
11 - Marche
12 - Molise
13 - Piemonte
14 - Puglia
15 - Sardegna
16 - Sicilia
17 - Toscana
18 - Trento (P.A.)
19 - Umbria
20 - Valle d'Aosta
21 - Veneto

Codice Provincia (Storico) can be ignored, it's used by ISTAT to keep tracks of cities in historical data

Codice Provincia/Uts is the one in use, and there are currently 107

In [133]:
cities_2020['Codice Provincia/Uts'].value_counts()
cities_2020.info()

<class 'pandas.DataFrame'>
RangeIndex: 7903 entries, 0 to 7902
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   Codice Ripartizione geografica  7903 non-null   int64
 1   Codice Regione                  7903 non-null   int64
 2   Codice Provincia (Storico)      7903 non-null   int64
 3   Codice Provincia/Uts            7903 non-null   int64
 4   Codice Comune (alfanumerico)    7903 non-null   int64
 5   Codice Comune (numerico)        7903 non-null   int64
 6   Comune                          7902 non-null   str  
 7   Comune (dizione straniera)      124 non-null    str  
 8   Sigla automobilistica           7811 non-null   str  
 9   Capoluogo di Provincia/Uts      7903 non-null   int64
 10  Capoluogo di Regione            7903 non-null   int64
 11  Popolazione legale              7903 non-null   int64
 12  Anno Censimento                 7903 non-null   int64
 13  Superficie (Km

For the Codice Comune (alfanumerico) and Codice Comune (numerico) both of them are identical

Now that we have a clear insight of what each column actually does, let's remove some columns and download the rest of the data, since the file for 'comuni_italiani.csv' only has entries for 2020. This is because we need for each year the updated number of legal residents to make a correct analysis.

In [181]:
polished_car_accident = car_accident[['REF_AREA', 'DATA_TYPE', 'RESULT', 'TIME_PERIOD', 'OBS_VALUE']]
polished_cities_2020 = cities_2020.iloc[:, [1,3,5,6,11,14,13]]

#Let's do the same for the other data, from 2021 to 2022
cities_2021 = pd.read_csv('data/comuni_italiani_2021.csv', sep=';')
cities_2022 = pd.read_csv('data/comuni_italiani_2022.csv', sep=';')
cities_2023 = pd.read_csv('data/comuni_italiani_2023.csv', sep=';')
cities_2024 = pd.read_csv('data/comuni_italiani_2024.csv', sep=';')

polished_cities_2021 = cities_2021.iloc[:, [5,14,11]]
polished_cities_2022 = cities_2022.iloc[:, [5,14,11]]
polished_cities_2023 = cities_2023.iloc[:, [5,14,11]]
polished_cities_2024 = cities_2024.iloc[:, [5,14,11]]


concat_cities = [polished_cities_2020, polished_cities_2021, polished_cities_2022, polished_cities_2023, polished_cities_2024]
polished_cities_complete = pd.concat(concat_cities).iloc[:,[2,5,4]].rename(columns={'Codice Comune (numerico)':'Codice Comune', 'Anno (Superficie)': 'Anno'})


Now that we have isolated the data we need for our analysis let's merge it togheter in a single csv file and save it

In [183]:
polished_car_accident.head(1)

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,1001,KILLINJ,F,2020,4


In [189]:
complete_dataset = polished_car_accident.merge(
    polished_cities_complete,
    left_on=['REF_AREA', 'TIME_PERIOD'],
    right_on=['Codice Comune', 'Anno']
)
complete_dataset.head(15)

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,Codice Comune,Anno,Popolazione legale
0,1001,KILLINJ,F,2020,4,1001,2020,2644
1,1001,KILLINJ,F,2021,2,1001,2021,2562
2,1001,KILLINJ,F,2022,5,1001,2022,2562
3,1001,KILLINJ,F,2023,6,1001,2023,2562
4,1001,KILLINJ,F,2024,5,1001,2024,2562
5,1001,KILLINJ,M,2020,0,1001,2020,2644
6,1001,KILLINJ,M,2021,0,1001,2021,2562
7,1001,KILLINJ,M,2022,0,1001,2022,2562
8,1001,KILLINJ,M,2023,1,1001,2023,2562
9,1001,KILLINJ,M,2024,0,1001,2024,2562


Now that the cleanup is completed, let's proceed with saving the whole database in a new file

In [190]:
complete_dataset.to_csv('data/complete_dataset.csv')